# Airbnb calendar Bronze ingestion (F06.2)

Ingests the Toronto Inside Airbnb `calendar.csv.gz` into its own Bronze
data product (`airbnb_calendar_bronze`) - separate from listings and
reviews - reading the immutable F04 manifest landing object.

Every value stays a raw string: no casting of dates, prices or
availability (that is Silver, F09.1). Rows failing the contract's
required-field/date checks are RETAINED with an explicit
`_quality_status` / `_quality_reason`, never dropped.

The manifest record is resolved by explicit `source_file_id`: the
family/entity/version identity is shared by the listings, calendar and
reviews files for Toronto 2026-09-16, so resolving by identity alone
is ambiguous (verified in F06.1).

Reruns are guarded against the Bronze slice itself (F06.3 will add
per-dataset control-table rows once the file-type key collision is
resolved): an exact same-file rerun skips; a corrected file (different
`source_file_id`) is blocked unless `force_reload=true`.

In [0]:
# Run-mode controls (F02.2 / F05.3 conventions, applied to the calendar
# dataset in F06.2-T02).
#
# run_mode defaults to "test": every table this notebook touches is
# resolved to an isolated *_test Delta table through
# src.common.environment. "live" resolves the development tables and is
# used only after the test-mode evidence gate has passed (and, for new
# datasets, the licence confirmation).
#
# force_reload is the explicit reprocess authorization: it permits an
# exact rerun of an already-ingested file and the controlled replacement
# of a corrected source file (a different source_file_id).
#
# source_file_id pins the EXACT landed manifest record this run reads.
# The family/entity/version identity (insideairbnb/toronto/2026-09-16 +
# csv.gz) is shared by the listings, calendar and reviews files, so
# resolving by identity alone is ambiguous - the explicit id is
# required (verified in F06.1).

dbutils.widgets.dropdown("run_mode", "test", ["test", "live"])
dbutils.widgets.text("force_reload", "false")
dbutils.widgets.text(
    "source_file_id",
    "09a7ef20b005bc7fb5ee0abdb6ba27f07a2d2e72a62885351442702ef38fefb2",
)

RUN_MODE = dbutils.widgets.get("run_mode")
FORCE_RELOAD = dbutils.widgets.get("force_reload")
SOURCE_FILE_ID = dbutils.widgets.get("source_file_id")

print(f"Run mode: {RUN_MODE} | force_reload: {FORCE_RELOAD}")
print(f"source_file_id: {SOURCE_FILE_ID}")

In [0]:
# Resolve every table through the run mode (F02.2) and load the ACTUAL
# Bronze contract for this dataset (F06.2-T02): no guessed column names.
from datetime import UTC, datetime
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [start, *start.parents]
        if (p / "src").is_dir() and (p / "config").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Cannot locate the TripPilot repository root (src/ and config/).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import functions as F

from src.airbnb.ingestion import write_snapshot
from src.airbnb.row_quality import annotate_row_quality, extract_row_validation_policy
from src.airbnb.source_inspection import (
    check_required_columns,
    inspect_csv_gz_header,
    verify_gzip_integrity,
)
from src.common.environment import (
    resolve_table,
    validate_force_reload,
    validate_run_mode,
)
from src.common.provenance import build_provenance_columns, build_run_id
from src.common.run_controls import (
    decide_ingestion_action,
    latest_successful_source_file_id,
    record_control_failure,
)
from src.common.source_acquisition import verify_landed_record
from src.common.source_manifest_store import ManifestStore
from src.quality.contract_loader import get_table_rules, load_contract

# Dataset identity for the calendar path (F06.1 evidence).
CITY = "toronto"
SNAPSHOT_DATE = "2026-09-16"
SOURCE = "insideairbnb"
SCHEMA_LABEL = "calendar"
CONTRACT_PATH = ROOT / "contracts" / "airbnb" / "bronze_calendar.yml"
RULES_KEY = "bronze_calendar"

RUN_MODE = validate_run_mode(RUN_MODE)
FORCE_RELOAD = validate_force_reload(FORCE_RELOAD)

# Every table name flows through the run mode: the manifest (test ->
# ops.source_file_manifest_test) and this dataset's separate Bronze
# destination (test -> bronze.airbnb_calendar_bronze_test). The Bronze
# product is NEVER shared with listings or reviews.
MANIFEST_TABLE = resolve_table("source_manifest", RUN_MODE)
BRONZE_TABLE = resolve_table("calendar_bronze", RUN_MODE)

# F06.3: this dataset's own ingestion-control table - separate from
# the listings control table, whose schema and history stay untouched.
CONTROL_TABLE = resolve_table("calendar_control", RUN_MODE)
DATASET = "calendar"

contract = load_contract(CONTRACT_PATH)
rules = get_table_rules(contract, RULES_KEY)
REQUIRED_COLUMNS = list(rules["required_columns"])
ROW_REQUIRED_FIELDS, ROW_DATE_FIELDS = extract_row_validation_policy(contract, RULES_KEY)

print(f"Manifest table: {MANIFEST_TABLE}")
print(f"Bronze target:  {BRONZE_TABLE}")
print(f"Contract:       {CONTRACT_PATH.name} ({len(REQUIRED_COLUMNS)} required columns)")
print(f"Row validation:  required={ROW_REQUIRED_FIELDS} dates={ROW_DATE_FIELDS}")

# Resolve and validate the landed source record (F06.2-T02)

Bronze reads ONLY the immutable F04 landing object recorded in the
manifest, resolved by the explicit `source_file_id` widget - never a
root or guessed Volume path, and never the shared family/entity/version
identity (ambiguous across the three Toronto files).

Before Spark parses anything, the landed bytes are verified against the
manifest (hash + size), the whole gzip stream must decompress cleanly,
the observed header must match the fingerprint recorded at registration
(schema-drift gate, F05.3), and every contract-required column must be
present.

In [0]:
# Resolve the exact landed record and validate the immutable landing
# object BEFORE Spark parses it.
store = ManifestStore(MANIFEST_TABLE, spark=spark)
source_record = store.read_landed(SOURCE_FILE_ID)

# Hash/size verification of the landed bytes (raises on any mismatch;
# a mismatch is an integrity incident, never an overwrite).
verify_landed_record(source_record)

# Full gzip decompression + exact header inspection of the landed bytes.
verify_gzip_integrity(source_record["landing_path"])
inspection = inspect_csv_gz_header(
    source_record["landing_path"], schema_label=SCHEMA_LABEL
)

# Schema-drift gate: the observed header fingerprint must equal the
# schema_version recorded at registration.
assert inspection["schema_version"] == source_record["schema_version"], (
    f"Schema drift: manifest recorded {source_record['schema_version']}, "
    f"landed header is {inspection['schema_version']}"
)

# Contract enforcement: every required column must be present.
check_required_columns(inspection["columns"], REQUIRED_COLUMNS)

print(f"Landing path:   {source_record['landing_path']}")
print(f"Byte size:       {source_record['byte_size']:,}")
print(f"File hash:       {source_record['file_hash_sha256'][:16]}...")
print(f"Schema version:  {source_record['schema_version']}")
print(f"Observed columns ({len(inspection['columns'])}): {inspection['columns']}")

# Ingestion-control decision (F06.3-T01)

Calendar records its run outcomes in its OWN ingestion-control table
(`airbnb_calendar_ingestion_control`): the listings control table and
its history stay untouched, and a listings or reviews row can never
influence this decision. Every row is tagged `dataset='calendar'` and
carries the exact `source_file_id` it processed (plus the identity a
controlled reprocess replaced).

The decision reuses the tested F05.3 helper, now dataset-scoped:

* no successful calendar load for this city/snapshot -> **load** (a
  first run or a new snapshot date, its own slice)
* the successful load used the SAME `source_file_id` -> **skip** (an
  exact retry is a safe no-op) unless `force_reload` authorizes an
  exact rerun
* a successful load used a DIFFERENT identity (corrected file) ->
  **blocked** with an explicit error unless `force_reload` authorizes
  the replacement

In [0]:
# F06.3-T01: dataset-scoped ingestion-control decision.
#
# Calendar has its OWN control table: the listings control table and
# its history stay untouched, and a listings or reviews row can never
# influence this decision. Rows are tagged dataset='calendar', so the
# scope is explicit on every row; legacy untagged rows count as
# listings history and never leak into this scope.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
      city STRING,
      snapshot_date STRING,
      run_id STRING,
      status STRING,
      row_count LONG,
      started_at_utc STRING,
      completed_at_utc STRING,
      error_message STRING,
      source_file_id STRING,
      prior_source_file_id STRING,
      dataset STRING
    ) USING DELTA
""")

existing_controls = [
    row.asDict()
    for row in spark.table(CONTROL_TABLE)
    .filter(
        (F.col("city") == CITY)
        & (F.col("snapshot_date") == SNAPSHOT_DATE)
    )
    .collect()
]

# Decide from the source-file identity within this dataset's scope:
# exact retry -> skip; corrected file -> blocked unless force_reload
# authorizes reprocess; no successful load or new snapshot -> load.
ingestion_action = decide_ingestion_action(
    existing_controls=existing_controls,
    city=CITY,
    snapshot_date=SNAPSHOT_DATE,
    source_file_id=source_record["source_file_id"],
    force_reload=FORCE_RELOAD,
    dataset=DATASET,
)

print(f"Control table: {CONTROL_TABLE}")
print(
    f"Existing control records for {DATASET}/{CITY}/{SNAPSHOT_DATE}: "
    f"{len(existing_controls)}"
)
print(f"Ingestion action: {ingestion_action}")

# Safe no-op: an exact retry of the same file. Nothing is read or
# written. A corrected file without force_reload already raised above
# with instructions; a deliberate rerun needs force_reload=true.
assert ingestion_action == "load", (
    f"Skipping ingestion: the exact source file "
    f"{source_record['source_file_id']} already loaded successfully for "
    f"{DATASET} / {CITY} / {SNAPSHOT_DATE}. Nothing was read or written. "
    "Set force_reload=true to rerun the identical file, or select the new "
    "source_file_id to reprocess a corrected file."
)

# Start the control record

The run is recorded before any data work, tagged with its dataset,
linked to its exact source-file identity - and, when reprocessing a
corrected file, to the identity it replaces, so the replacement is
auditable (the F05.3 pattern, applied per dataset).

In [0]:
# Create a unique ID for this pipeline execution.
run_id = build_run_id()

# Link this run to its exact source-file identity, and - when
# reprocessing a corrected file - to the identity it replaced.
prior_source_file_id = latest_successful_source_file_id(existing_controls)

started_at_utc = datetime.now(UTC).isoformat()

spark.sql(f"""
    INSERT INTO {CONTROL_TABLE}
    VALUES (
        '{CITY}',
        '{SNAPSHOT_DATE}',
        '{run_id}',
        'running',
        0,
        '{started_at_utc}',
        NULL,
        NULL,
        '{source_record['source_file_id']}',
        {"'" + prior_source_file_id + "'" if prior_source_file_id else "NULL"},
        '{DATASET}'
    )
""")

print(f"Started ingestion run: {run_id}")
if prior_source_file_id and prior_source_file_id != source_record["source_file_id"]:
    print(f"Controlled reprocess: replacing source_file_id {prior_source_file_id}")

# Read the landed source file (F06.2-T01)

All source values stay STRINGS: header on, schema inference off, no
casting of dates, prices or availability - normalization belongs to
Silver (F09.1). `multiLine` + escaped quotes keep any quoted field
intact.

In [0]:
# A malformed CSV that Spark cannot parse is recorded as a failed
# control run before anything is written (F06.3-T02).
try:
    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(source_record["landing_path"])
    )

    # Count input rows now so we can compare them with Bronze output later.
    raw_row_count = raw_df.count()
except Exception as read_error:
    record_control_failure(spark, CONTROL_TABLE, run_id, read_error)
    raise

print(f"Input path: {source_record['landing_path']}")
print(f"Raw rows read: {raw_row_count:,}")
display(raw_df.limit(5))

# Row quality (F06.2-T03)

Required identifier/date fields come from the contract, never
hard-coded. Rows failing a check are RETAINED in Bronze with an
explicit `_quality_reason` and `_quality_status` - never dropped,
quarantined or coerced.

In [0]:
quality_df = annotate_row_quality(raw_df, ROW_REQUIRED_FIELDS, ROW_DATE_FIELDS)

status_counts = {
    row["_quality_status"]: row["count"]
    for row in quality_df.groupBy("_quality_status").count().collect()
}
valid_count = status_counts.get("valid", 0)
invalid_count = status_counts.get("invalid", 0)

print(f"Valid rows:   {valid_count:,}")
print(f"Invalid rows: {invalid_count:,} (retained with _quality_reason)")

if invalid_count:
    display(quality_df.filter(F.col("_quality_status") == "invalid").limit(20))

# Provenance metadata

Every Bronze row becomes traceable to its source file, city, snapshot
date, pipeline execution, manifest identity and content hash - the 8
provenance columns required by the calendar contract.

In [0]:
# run_id was assigned when this run's control record started (F06.3).
provenance = build_provenance_columns(
    source=SOURCE,
    city=CITY,
    snapshot_date=SNAPSHOT_DATE,
    raw_path=source_record["landing_path"],
    run_id=run_id,
    source_file_id=source_record["source_file_id"],
    file_hash_sha256=source_record["file_hash_sha256"],
)

bronze_df = (
    quality_df
    .withColumn("_source", F.lit(provenance["source"]))
    .withColumn("_city", F.lit(provenance["city"]))
    .withColumn("_snapshot_date", F.lit(provenance["snapshot_date"]))
    .withColumn("_raw_path", F.lit(provenance["raw_path"]))
    .withColumn("_run_id", F.lit(provenance["run_id"]))
    .withColumn("_ingested_at_utc", F.lit(provenance["ingested_at_utc"]))
    .withColumn("_source_file_id", F.lit(provenance["source_file_id"]))
    .withColumn("_file_hash_sha256", F.lit(provenance["file_hash_sha256"]))
)

print(f"Run ID: {run_id}")
display(bronze_df.limit(5))

# Replace only the snapshot slice

`write_snapshot` is the single tested write path: one atomic Delta
overwrite with `replaceWhere`, so the slice is never observed
half-written. `merge_schema` lets the table pick up the provenance and
quality columns on first creation.

In [0]:
# A write failure is recorded on this run's control row before the
# exception propagates (F06.3-T02).
try:
    write_snapshot(
        bronze_df,
        BRONZE_TABLE,
        CITY,
        SNAPSHOT_DATE,
        "_city",
        "_snapshot_date",
        merge_schema=True,
    )

    bronze_row_count = (
        spark.table(BRONZE_TABLE)
        .filter((F.col("_city") == CITY) & (F.col("_snapshot_date") == SNAPSHOT_DATE))
        .count()
    )
except Exception as write_error:
    record_control_failure(spark, CONTROL_TABLE, run_id, write_error)
    raise

print(f"Target table: {BRONZE_TABLE}")
print(f"Bronze rows written for {CITY} / {SNAPSHOT_DATE}: {bronze_row_count:,}")

# Reconcile and verify

Every row for this source identity must match the parsed source count
and come from exactly one run. Invalid rows are counted in-table so
the retention promise is auditable.

In [0]:
# Record completion on this run's control row (F06.3-T02): the
# per-file outcome for this dataset - status, row count, timestamp.
completed_at_utc = datetime.now(UTC).isoformat()
spark.sql(f"""
    UPDATE {CONTROL_TABLE}
    SET status = 'success',
        row_count = {bronze_row_count},
        completed_at_utc = '{completed_at_utc}',
        error_message = NULL
    WHERE run_id = '{run_id}'
""")

verification_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_source_file_id") == source_record["source_file_id"])
    .agg(
        F.count("*").alias("bronze_row_count"),
        F.countDistinct("_run_id").alias("distinct_run_ids"),
        F.sum(F.when(F.col("_quality_status") == "invalid", 1).otherwise(0)).alias(
            "invalid_rows"
        ),
    )
)
display(verification_df)

identity_counts = verification_df.collect()[0]
assert identity_counts["distinct_run_ids"] == 1, (
    f"Expected one run identity, found {identity_counts['distinct_run_ids']}"
)
assert bronze_row_count == raw_row_count, (
    f"Row-count mismatch: raw={raw_row_count}, bronze={bronze_row_count}"
)
assert identity_counts["bronze_row_count"] == raw_row_count, (
    f"Source-identity mismatch: raw={raw_row_count}, "
    f"bronze for this source_file_id={identity_counts['bronze_row_count']}"
)
assert identity_counts["invalid_rows"] == invalid_count, (
    "Invalid-row count changed between annotation and the Bronze write"
)

print("Calendar Bronze ingestion verification passed.")

In [0]:
# Post-run summary of every loaded city/snapshot slice, including the
# retained invalid rows (F06.2-T03).
display(
    spark.sql(f"""
        SELECT
            _city,
            _snapshot_date,
            COUNT(*) AS row_count,
            SUM(CASE WHEN _quality_status = 'invalid' THEN 1 ELSE 0 END) AS invalid_rows,
            COUNT(DISTINCT _run_id) AS run_count,
            COUNT(DISTINCT _source_file_id) AS source_file_count,
            MAX(_source_file_id) AS current_source_file_id
        FROM {BRONZE_TABLE}
        GROUP BY _city, _snapshot_date
    """)
)